# Ecoy — photography to Shopify Files

Uploads the web-optimised photography library from the shared Drive to Shopify Files, so every
image gets a permanent CDN url that Claude Design and any template can reference directly.

**Because filenames are unique, the url is predictable:**

```
https://cdn.shopify.com/s/files/1/0498/6100/1367/files/BambooSheetSet-Solid-Dusk-High45-NoTalent-Real-01.webp
```

## Before you run it

The notebook mints its own access token with the **client credentials grant**, so there is no
token to paste and none to rotate. It needs the app's two credentials, which you'll find under
**App settings → Credentials** in the Dev Dashboard.

1. In Colab, click the **key icon** in the left sidebar and add two secrets, both with
   *Notebook access* switched on:
   - `SHOPIFY_CLIENT_ID`
   - `SHOPIFY_CLIENT_SECRET`
2. Run the cells in order. Leave `LIMIT = 100` for the first run.

The app must be **installed on the store**, and the store must be in the same Shopify
organization as the app, or the token request fails with `shop_not_permitted`.

Safe to re-run: anything already in Shopify Files is skipped, so a failed run costs one batch.


In [ ]:
# ---- config -------------------------------------------------------------
LIMIT        = 100          # None = the whole manifest. Keep 100 for the first run.
SHOP         = 'happy-earth-australia.myshopify.com'   # the myshopify domain, not the storefront domain
API_VERSION  = '2026-07'
BATCH        = 25           # files per staged-upload round; API hard limit is 250

DRIVE        = '/content/drive/Shareddrives/Ecoy shared drive'
WEB_ROOT     = f'{DRIVE}/Photography - Web'
MANIFEST     = f'{DRIVE}/Photography - Web/shopify-upload-manifest.json'
RESULTS      = f'{DRIVE}/Photography - Web/shopify-upload-results.json'


## 1. Mount Drive and load the manifest


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=False)

import json, os, time, pathlib
import requests

manifest = json.load(open(MANIFEST))
files = manifest['files']
print(f"manifest: {len(files)} files")

CLIENT_ID     = userdata.get('SHOPIFY_CLIENT_ID')
CLIENT_SECRET = userdata.get('SHOPIFY_CLIENT_SECRET')
assert CLIENT_ID and CLIENT_SECRET, \
    'Add SHOPIFY_CLIENT_ID and SHOPIFY_CLIENT_SECRET in the Colab secrets panel (key icon)'

ENDPOINT = f'https://{SHOP}/admin/api/{API_VERSION}/graphql.json'
_token = {'value': None, 'expires': 0}


def token():
    """Mint an access token, reusing it until five minutes before it expires.

    Client credentials tokens last 24 hours, which outlives any single run, but a
    6,350-file upload is long enough that refreshing is worth the few lines.
    """
    if _token['value'] and time.time() < _token['expires']:
        return _token['value']
    r = requests.post(f'https://{SHOP}/admin/oauth/access_token',
                      headers={'Content-Type': 'application/x-www-form-urlencoded'},
                      data={'grant_type': 'client_credentials',
                            'client_id': CLIENT_ID, 'client_secret': CLIENT_SECRET},
                      timeout=60)
    if r.status_code != 200:
        body = r.text[:300]
        if 'shop_not_permitted' in body:
            raise RuntimeError(
                'shop_not_permitted: the app and the store must be in the same Shopify '
                'organization, and the app must be installed on the store. Check Installs '
                'on the app home page in the Dev Dashboard.')
        raise RuntimeError(f'token request failed ({r.status_code}): {body}')
    d = r.json()
    _token['value'] = d['access_token']
    _token['expires'] = time.time() + d.get('expires_in', 86399) - 300
    print(f"token minted, scopes: {d.get('scope')}")
    return _token['value']


token()   # fail fast here rather than mid-upload


## 2. GraphQL helper

Retries on `THROTTLED` by waiting for the leaky bucket to refill. Shopify Plus restores
1,000 cost points a second and a mutation costs 10, so throttling should be rare.


In [ ]:
def gql(query, variables=None, tries=5):
    for attempt in range(tries):
        r = requests.post(ENDPOINT,
                          headers={'X-Shopify-Access-Token': token(),
                                   'Content-Type': 'application/json'},
                          json={'query': query, 'variables': variables or {}}, timeout=120)
        r.raise_for_status()
        body = r.json()
        errs = body.get('errors') or []
        if any((e.get('extensions') or {}).get('code') == 'THROTTLED' for e in errs):
            wait = 2 ** attempt
            print(f'   throttled, waiting {wait}s'); time.sleep(wait); continue
        if errs:
            raise RuntimeError(errs)
        return body['data']
    raise RuntimeError('still throttled after retries')

print(gql('{ shop { name } }')['shop']['name'], '- connected')


## 3. What is already uploaded

Pages every existing file once so a re-run skips them. This is what makes the notebook
idempotent: run it after each shoot and only new photography uploads.


In [ ]:
def existing_filenames():
    names, cursor = set(), None
    q = '''query($cursor: String) { files(first: 250, after: $cursor) {
             pageInfo { hasNextPage endCursor }
             edges { node { ... on MediaImage { image { url } } } } } }'''
    while True:
        d = gql(q, {'cursor': cursor})['files']
        for e in d['edges']:
            img = (e['node'] or {}).get('image') or {}
            if img.get('url'):
                names.add(img['url'].split('/')[-1].split('?')[0])
        if not d['pageInfo']['hasNextPage']: break
        cursor = d['pageInfo']['endCursor']
    return names

already = existing_filenames()
print(f'{len(already)} files already in Shopify Files')

todo = [f for f in files if f['name'] not in already]
if LIMIT: todo = todo[:LIMIT]
print(f'{len(todo)} to upload this run')


## 4. Upload

Three steps per batch: ask Shopify for staged upload targets, POST the bytes straight to
their storage (this does not touch the API rate limit), then register each file.


In [ ]:
STAGED = '''mutation($input: [StagedUploadInput!]!) {
  stagedUploadsCreate(input: $input) {
    stagedTargets { url resourceUrl parameters { name value } }
    userErrors { field message } } }'''

CREATE = '''mutation($files: [FileCreateInput!]!) {
  fileCreate(files: $files) {
    files { id fileStatus alt ... on MediaImage { image { url } } }
    userErrors { field message } } }'''

results, failures = [], []

for i in range(0, len(todo), BATCH):
    batch = todo[i:i + BATCH]
    paths = [f"{WEB_ROOT}/{f['web']}" for f in batch]
    keep  = [(f, p) for f, p in zip(batch, paths) if os.path.exists(p)]
    for f, p in zip(batch, paths):
        if not os.path.exists(p):
            failures.append({'name': f['name'], 'why': 'not found in Drive', 'path': p})
    if not keep: continue

    inp = [{'filename': f['name'], 'mimeType': 'image/webp', 'resource': 'FILE',
            'httpMethod': 'POST', 'fileSize': str(os.path.getsize(p))} for f, p in keep]
    targets = gql(STAGED, {'input': inp})['stagedUploadsCreate']
    if targets['userErrors']:
        failures.append({'batch': i, 'why': targets['userErrors']}); continue

    sources = []
    for (f, p), t in zip(keep, targets['stagedTargets']):
        form = [(param['name'], (None, param['value'])) for param in t['parameters']]
        with open(p, 'rb') as fh:
            form.append(('file', (f['name'], fh, 'image/webp')))
            up = requests.post(t['url'], files=form, timeout=300)
        if up.status_code not in (200, 201, 204):
            failures.append({'name': f['name'], 'why': f'staged upload HTTP {up.status_code}'})
            continue
        alt = ' '.join(str(f[k]) for k in ('product','colour','angle') if f.get(k))
        sources.append({'originalSource': t['resourceUrl'], 'contentType': 'IMAGE', 'alt': alt})

    if not sources: continue
    created = gql(CREATE, {'files': sources})['fileCreate']
    if created['userErrors']:
        failures.append({'batch': i, 'why': created['userErrors']})
    results.extend(created['files'])
    print(f'  {min(i + BATCH, len(todo))}/{len(todo)} uploaded')

print(f'\ndone: {len(results)} registered, {len(failures)} failures')


## 5. Wait for processing, then capture the real urls

Shopify processes asynchronously. This polls until every file is `READY` and records the url
Shopify actually assigned — which is the url to trust, not the one we assumed.


In [ ]:
ids = [f['id'] for f in results if f.get('id')]
urls, pending = {}, list(ids)
Q = '''query($ids: [ID!]!) { nodes(ids: $ids) {
         ... on MediaImage { id fileStatus image { url width height } } } }'''

for attempt in range(20):
    if not pending: break
    nodes, nxt = [], []
    for j in range(0, len(pending), 100):
        nodes += gql(Q, {'ids': pending[j:j+100]})['nodes']
    for n in nodes:
        if not n: continue
        if n.get('fileStatus') == 'READY' and (n.get('image') or {}).get('url'):
            urls[n['id']] = n['image']['url']
        else:
            nxt.append(n['id'])
    pending = nxt
    if pending:
        print(f'   {len(pending)} still processing'); time.sleep(5)

print(f'{len(urls)} ready, {len(pending)} still pending')


## 6. Verify every url actually resolves

Catches the silent failure mode: Shopify renaming a file on a collision, so the url you
predicted 404s. If `renamed` is anything other than zero, tell Claude.


In [ ]:
import concurrent.futures as cf
PATTERN = manifest['cdnPattern']

def check(item):
    name, url = item
    predicted = PATTERN.format(name=name)
    ok = requests.head(url.split('?')[0], timeout=30).status_code == 200
    return {'name': name, 'url': url, 'predictable': url.split('?')[0] == predicted, 'resolves': ok}

by_name = {}
for fid, url in urls.items():
    by_name[url.split('/')[-1].split('?')[0]] = url

with cf.ThreadPoolExecutor(max_workers=16) as ex:
    checked = list(ex.map(check, by_name.items()))

renamed  = [c for c in checked if not c['predictable']]
broken   = [c for c in checked if not c['resolves']]
print(f'checked {len(checked)}:  {len(renamed)} renamed by Shopify,  {len(broken)} do not resolve')
for c in renamed[:10]: print('   renamed:', c['name'])
for c in broken[:10]:  print('   broken :', c['url'])


## 7. Write the results back to Drive


In [ ]:
out = {'shop': SHOP, 'ranAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
       'uploaded': len(urls), 'failures': failures,
       'renamedByShopify': renamed, 'broken': broken,
       'urls': {c['name']: c['url'] for c in checked}}

prev = {}
if os.path.exists(RESULTS):
    prev = json.load(open(RESULTS)).get('urls', {})
out['urls'] = {**prev, **out['urls']}

with open(RESULTS, 'w') as fh:
    json.dump(out, fh, indent=1)
print(f"wrote {len(out['urls'])} urls to {RESULTS}")
print('\nSpot-check one in a browser:')
for name, url in list(out['urls'].items())[:3]: print('  ', url)
